In [3]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression

df_pf = pd.read_csv('/home/fabric/work/qkd-pqc-dependability/results/joint_fault_pf_results.csv')
df_pf['loss_pct'] = df_pf['loss_pct'].fillna(0)

for c in ['qber', 'key_bits_per_sec']:
    df_pf[c] = pd.to_numeric(df_pf[c], errors='coerce')

def label_failure(row, keyrate_thresh=0.0):
    qber_fail = row['qber'] >= 0.11
    rate_fail = row['key_bits_per_sec'] < keyrate_thresh
    return int(qber_fail or rate_fail)

df_pf['Y'] = df_pf.apply(label_failure, axis=1)

print("Label distribution:")
print(df_pf['Y'].value_counts())
print("\nWhich rows are labeled failure:")
print(df_pf[df_pf['Y']==1][['condition', 'loss_pct', 'polarization_fidelity', 'qber', 'key_bits_per_sec']])

# Fit on polarization_fidelity + loss_pct
X = df_pf[['polarization_fidelity', 'loss_pct']].values
y = df_pf['Y'].values
clf = LogisticRegression()
clf.fit(X, y)

# Score a denser grid than what you've actually run
pf_grid = np.linspace(0.75, 1.0, 30)
loss_grid = np.linspace(0, 20, 30)
grid_points = np.array([[pf, l] for pf in pf_grid for l in loss_grid])
probs = clf.predict_proba(grid_points)[:, 1]

uncertainty = np.abs(probs - 0.5)
top_uncertain_idx = np.argsort(uncertainty)[:10]
next_queries = grid_points[top_uncertain_idx]

print("\nTop 10 most uncertain (polarization_fidelity, loss_pct) points to query next:")
for pt, p in zip(next_queries, probs[top_uncertain_idx]):
    print(f"  pf={pt[0]:.3f}, loss_pct={pt[1]:.2f}, P(failure)={p:.3f}")

Label distribution:
Y
0    48
1     4
Name: count, dtype: int64

Which rows are labeled failure:
     condition  loss_pct  polarization_fidelity      qber  key_bits_per_sec
3   loss_15pct      15.0                    0.8  0.117188               0.0
10   loss_5pct       5.0                    0.8  0.117188               0.0
36    baseline       0.0                    0.8  0.114883               0.0
48    baseline       0.0                    0.8  0.113111               0.0

Top 10 most uncertain (polarization_fidelity, loss_pct) points to query next:
  pf=0.750, loss_pct=0.00, P(failure)=0.082
  pf=0.759, loss_pct=0.00, P(failure)=0.082
  pf=0.767, loss_pct=0.00, P(failure)=0.082
  pf=0.750, loss_pct=0.69, P(failure)=0.082
  pf=0.776, loss_pct=0.00, P(failure)=0.082
  pf=0.759, loss_pct=0.69, P(failure)=0.082
  pf=0.784, loss_pct=0.00, P(failure)=0.081
  pf=0.767, loss_pct=0.69, P(failure)=0.081
  pf=0.750, loss_pct=1.38, P(failure)=0.081
  pf=0.793, loss_pct=0.00, P(failure)=0.081


In [4]:
df_pf['dist_to_threshold'] = np.abs(df_pf['qber'] - 0.11)
print(df_pf.sort_values('dist_to_threshold')[
    ['condition', 'loss_pct', 'polarization_fidelity', 'qber', 'key_bits_per_sec', 'dist_to_threshold']
].head(10))

     condition  loss_pct  polarization_fidelity      qber  key_bits_per_sec  \
6    loss_5pct       5.0                    0.8  0.109661          0.129506   
48    baseline       0.0                    0.8  0.113111          0.000000   
39  loss_15pct      15.0                    0.8  0.106632          2.507482   
38   loss_5pct       5.0                    0.8  0.106632          2.507482   
36    baseline       0.0                    0.8  0.114883          0.000000   
11  loss_15pct      15.0                    0.8  0.104712          1.780894   
50   loss_5pct       5.0                    0.8  0.102902          2.623642   
10   loss_5pct       5.0                    0.8  0.117188          0.000000   
3   loss_15pct      15.0                    0.8  0.117188          0.000000   
43  loss_15pct      15.0                    0.8  0.098573          8.804941   

    dist_to_threshold  
6            0.000339  
48           0.003111  
39           0.003368  
38           0.003368  
36        

In [20]:
SLICE_NAME = 'qfabric-bb84-2'
SCENARIO   = 'validation/scenarios/fabric_1km.yml'

In [21]:
import os, sys, json
from pathlib import Path

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_DIR)); sys.path.insert(0, str(PROJECT_DIR / 'scripts'))
import deploy_fabric_modified as df

fablib = df.get_fablib()
slice_obj = fablib.get_slice(name=SLICE_NAME)
slice_obj.show()

Orchestrator,orchestrator.fabric-testbed.net
Credential Manager,cm.fabric-testbed.net
Core API,uis.fabric-testbed.net
Artifact Manager,artifacts.fabric-testbed.net
CEPH Manager,https://ceph-mgr.fabric-testbed.net
Token File,/home/fabric/work/fabric_config/id_token (3).json
Project ID,24f4c8f3-e872-492a-9a83-b48211a91966
Bastion Host,bastion.fabric-testbed.net
Bastion Username,audreyf_0000527467
Bastion Private Key File,/home/fabric/work/fabric_config/fabric_bastion_key
Slice Public Key File,/home/fabric/work/fabric_config/slice_key.pub


User: audreyf@illinois.edu bastion key is valid!
Configuration is valid


ID,d8ea8804-1e65-4186-a902-d6006de32667
Name,qfabric-bb84-2
Lease Expiration (UTC),2026-07-28 21:27:56 +0000
Lease Start (UTC),2026-07-20 21:27:56 +0000
Project ID,24f4c8f3-e872-492a-9a83-b48211a91966
State,StableOK
Email,audreyf@illinois.edu
UserId,8616dd8c-5b61-45db-84bb-791c21e82a89


ID,d8ea8804-1e65-4186-a902-d6006de32667
Name,qfabric-bb84-2
Lease Expiration (UTC),2026-07-28 21:27:56 +0000
Lease Start (UTC),2026-07-20 21:27:56 +0000
Project ID,24f4c8f3-e872-492a-9a83-b48211a91966
State,StableOK
Email,audreyf@illinois.edu
UserId,8616dd8c-5b61-45db-84bb-791c21e82a89


In [9]:
import os, csv, json
import deploy_fabric_modified as deploy

FIELDNAMES_3WAY = ['condition', 'loss_pct', 'qber', 'sifted_bits', 'final_key_bits',
                    'secure_key_rate', 'elapsed_seconds', 'key_bits_per_sec',
                    'polarization_fidelity', 'dark_count_rate', 'run', 'note']

def run_three_way_sweep(slice_obj, pf_values, dark_count_values, conditions, n_runs=2,
                          out_csv='/home/fabric/work/qkd-pqc-dependability/results/joint_fault_3way_results.csv'):
    all_rows = []
    total = len(pf_values) * len(dark_count_values) * len(conditions) * n_runs
    done = 0
    write_header = not os.path.exists(out_csv)

    for pf in pf_values:
        for dc in dark_count_values:
            scenario_content = f"""name: pf_{pf}_dc_{dc}
channel:
  distance_km: 10.0
  attenuation_db_per_km: 0.2
  polarization_fidelity: {pf}
detector:
  efficiency: 0.8
  dark_count_rate: {dc}
  dead_time: 0.0
  timing_jitter: 0.0
protocol:
  num_photons: 10000
  send_rate_hz: 10000.0
  sample_fraction: 0.1
  wavelength: 0
seed: 42
"""
            scenario_path = f'/home/fabric/work/qkd-pqc-dependability/validation/scenarios/temp_pf_{pf}_dc_{dc}.yml'
            with open(scenario_path, 'w') as f:
                f.write(scenario_content)

            for run in range(n_runs):
                rows = deploy.run_network_conditions_experiment(slice_obj, scenario_path, conditions=conditions)
                for row in rows:
                    row['polarization_fidelity'] = pf
                    row['dark_count_rate'] = dc
                    row['run'] = run + 1
                    all_rows.append(row)
                    with open(out_csv, 'a', newline='') as f:
                        writer = csv.DictWriter(f, fieldnames=FIELDNAMES_3WAY, extrasaction='ignore')
                        if write_header:
                            writer.writeheader()
                            write_header = False
                        writer.writerow(row)
                done += len(conditions)
                print(f"Progress: {done}/{total} runs complete")

    with open('/home/fabric/work/qkd-pqc-dependability/results/joint_fault_3way_results.json', 'w') as f:
        json.dump(all_rows, f, indent=2)
    return all_rows

# 3 levels per fault class, chosen to bracket where you already know PF's boundary sits
pf_values = [0.8, 0.9, 1.0]
dark_count_values = [10, 1000, 100000]
conditions = (
    [{'name': 'baseline'}] +
    [{'name': f'loss_{l}pct', 'loss_pct': l} for l in [1, 5, 15]]
)

rows_3way = run_three_way_sweep(slice_obj, pf_values, dark_count_values, conditions, n_runs=2)


##### [1/4] classical condition: baseline #####
  cleared classical netem on Alice and Bob

=== Running BB84 protocol ===
  Bob data-plane IP: 10.10.1.2 (for classical channel)
  Alice data iface:  enp7s0
  Bob data iface:    enp7s0
  Cleaning up previous runs...
  Ensuring deps on alice...
  Ensuring deps on bob...
  Starting Bob...
  Alice MAC args:  --dst-mac '1A:9E:9B:43:74:D7' --src-mac '0E:A9:B5:58:70:AB'
  Starting Alice...
  Waiting for BB84 to complete...
  Alice finished
  Alice output: Alice: Sending 10000 photons on enp7s0
Alice: Finished sending 10000 photons
Alice: Connecting to Bob at 10.10.1.2:5100
  Retrying connection to 10.10.1.2:5100 (attempt 2/60)...
  Retrying connection to 10.10.1.2:5100 (attempt 3/60)...
  Retrying connection to 10.10.1.2:5100 (attempt 4/60)...
  Retrying connection to 10.10.1.2:5100 (attempt 5/60)...
  Retrying connection to 10.10.1.2:5100 (attempt 6/60)...
  Retrying connection to 10.10.1.2:5100 (attempt 7/60)...
  Retrying connection to 10.1

In [16]:
import numpy as np
from sklearn.gaussian_process import GaussianProcessClassifier
from sklearn.gaussian_process.kernels import RBF

def fit_and_query(X_pool, y_pool, n_grid=15):
    # Normalize features so PF (~0.8-1.0), dark_count (~10-100,000), and loss (~0-15)
    # don't distort distance just because their raw scales differ wildly
    X_mean, X_std = X_pool.mean(axis=0), X_pool.std(axis=0)
    X_norm = (X_pool - X_mean) / X_std

    # Anisotropic kernel: one length-scale per dimension, tuned during fit()
    kernel = RBF(length_scale=[1.0, 1.0, 1.0])
    gp = GaussianProcessClassifier(kernel=kernel)
    gp.fit(X_norm, y_pool)   # <-- this is the actual "learning" step

    # Build a grid of candidate points, matching your real tested ranges
    pf_grid = np.linspace(0.80, 1.0, n_grid)
    dc_grid = np.logspace(1, 5, n_grid)
    loss_grid = np.linspace(0, 15, n_grid)
    candidates = np.array([[pf, dc, l] for pf in pf_grid for dc in dc_grid for l in loss_grid])
    candidates_norm = (candidates - X_mean) / X_std

    # Ask the trained model: for each candidate, what's P(failure)?
    probs = gp.predict_proba(candidates_norm)[:, 1]

    # "Most uncertain" = probability closest to 0.5 (a coin flip, in the model's view)
    uncertainty = np.abs(probs - 0.5)
    best_idx = np.argmin(uncertainty)
    return candidates[best_idx], probs[best_idx], probs, candidates, gp, X_mean, X_std

In [17]:
X_pool = df_3way[['polarization_fidelity', 'dark_count_rate', 'loss_pct']].values.astype(float)
y_pool = df_3way['Y'].values.astype(float)

next_point, p_fail, all_probs, all_candidates, gp, X_mean, X_std = fit_and_query(X_pool, y_pool)
print(f"Next query: pf={next_point[0]:.3f}, dark_count={next_point[1]:.0f}, loss_pct={next_point[2]:.2f} (P(fail)≈{p_fail:.3f})")

# Sanity check: does the model reproduce a pattern you already know is real?
test_points = np.array([[0.8, 10, l] for l in [0, 1, 5, 15]])
test_norm = (test_points - X_mean) / X_std
print("P(fail) at PF=0.8, dark_count=10, loss=[0,1,5,15]:", gp.predict_proba(test_norm)[:, 1])

Next query: pf=0.800, dark_count=100000, loss_pct=0.00 (P(fail)≈0.263)
P(fail) at PF=0.8, dark_count=10, loss=[0,1,5,15]: [0.26273738 0.26273737 0.26273736 0.26273737]


In [18]:
print(gp.kernel_)

RBF(length_scale=[1.61, 1.82e+03, 3.56e+03])


In [22]:
import os, csv, json

FIELDNAMES_EFFLOSS = ['condition', 'loss_pct', 'qber', 'sifted_bits', 'final_key_bits',
                        'secure_key_rate', 'elapsed_seconds', 'key_bits_per_sec',
                        'detector_efficiency', 'run', 'note']

def run_efficiency_loss_sweep(slice_obj, efficiency_values, conditions, n_runs_map,
                                out_csv='/home/fabric/work/qkd-pqc-dependability/results/joint_fault_efficiency_loss.csv'):
    """n_runs_map: dict mapping efficiency value -> n_runs, so you can concentrate
    replicates where you expect the boundary to matter most."""
    all_rows = []
    total = sum(len(conditions) * n_runs_map[eff] for eff in efficiency_values)
    done = 0
    write_header = not os.path.exists(out_csv)

    for eff in efficiency_values:
        n_runs = n_runs_map[eff]
        scenario_content = f"""name: pf0.8_eff_{eff}
channel:
  distance_km: 10.0
  attenuation_db_per_km: 0.2
  polarization_fidelity: 0.8
detector:
  efficiency: {eff}
  dark_count_rate: 10.0
  dead_time: 0.0
  timing_jitter: 0.0
protocol:
  num_photons: 10000
  send_rate_hz: 10000.0
  sample_fraction: 0.1
  wavelength: 0
seed: 42
"""
        scenario_path = f'/home/fabric/work/qkd-pqc-dependability/validation/scenarios/temp_pf08_eff_{eff}.yml'
        with open(scenario_path, 'w') as f:
            f.write(scenario_content)

        for run in range(n_runs):
            rows = deploy.run_network_conditions_experiment(slice_obj, scenario_path, conditions=conditions)
            for row in rows:
                row['detector_efficiency'] = eff
                row['run'] = run + 1
                all_rows.append(row)
                with open(out_csv, 'a', newline='') as f:
                    writer = csv.DictWriter(f, fieldnames=FIELDNAMES_EFFLOSS, extrasaction='ignore')
                    if write_header:
                        writer.writeheader()
                        write_header = False
                    writer.writerow(row)
            done += len(conditions)
            print(f"Progress: {done}/{total} runs complete")

    return all_rows

# Uneven replication: more depth at low efficiency (further from ideal, more likely to interact with loss)
efficiency_values = [0.5, 0.7, 0.85, 1.0]
n_runs_map = {0.5: 6, 0.7: 5, 0.85: 4, 1.0: 3}

conditions = (
    [{'name': 'baseline'}] +
    [{'name': f'loss_{l}pct', 'loss_pct': l} for l in [1, 5, 15]]
)

rows_effloss = run_efficiency_loss_sweep(slice_obj, efficiency_values, conditions, n_runs_map)


##### [1/4] classical condition: baseline #####
  cleared classical netem on Alice and Bob

=== Running BB84 protocol ===
  Bob data-plane IP: 10.10.1.2 (for classical channel)
  Alice data iface:  enp7s0
  Bob data iface:    enp7s0
  Cleaning up previous runs...
  Ensuring deps on alice...
  Ensuring deps on bob...
  Starting Bob...
  Alice MAC args:  --dst-mac '1A:9E:9B:43:74:D7' --src-mac '0E:A9:B5:58:70:AB'
  Starting Alice...
  Waiting for BB84 to complete...
  Alice finished
  Alice output: Alice: Sending 10000 photons on enp7s0
Alice: Finished sending 10000 photons
Alice: Connecting to Bob at 10.10.1.2:5100
  Retrying connection to 10.10.1.2:5100 (attempt 2/60)...
  Retrying connection to 10.10.1.2:5100 (attempt 3/60)...
  Retrying connection to 10.10.1.2:5100 (attempt 4/60)...
  Retrying connection to 10.10.1.2:5100 (attempt 5/60)...
  Retrying connection to 10.10.1.2:5100 (attempt 6/60)...
  Retrying connection to 10.10.1.2:5100 (attempt 7/60)...
  Retrying connection to 10.1

KeyboardInterrupt: 

In [ ]:
print("alive check")

In [23]:
efficiency_values_remaining = [0.7, 0.85, 1.0]
n_runs_map_remaining = {0.7: 4, 0.85: 4, 1.0: 3}  # 0.7 only needs 4 more (had 1, wanted 5)

rows_effloss_continued = run_efficiency_loss_sweep(slice_obj, efficiency_values_remaining, conditions, n_runs_map_remaining)


##### [1/4] classical condition: baseline #####
  cleared classical netem on Alice and Bob

=== Running BB84 protocol ===
  Bob data-plane IP: 10.10.1.2 (for classical channel)
  Alice data iface:  enp7s0
  Bob data iface:    enp7s0
  Cleaning up previous runs...
  Ensuring deps on alice...
  Ensuring deps on bob...
  Starting Bob...
  Alice MAC args:  --dst-mac '1A:9E:9B:43:74:D7' --src-mac '0E:A9:B5:58:70:AB'
  Starting Alice...
  Waiting for BB84 to complete...
  Alice finished
  Alice output: Alice: Sending 10000 photons on enp7s0
Alice: Finished sending 10000 photons
Alice: Connecting to Bob at 10.10.1.2:5100
  Retrying connection to 10.10.1.2:5100 (attempt 2/60)...
  Retrying connection to 10.10.1.2:5100 (attempt 3/60)...
  Retrying connection to 10.10.1.2:5100 (attempt 4/60)...
  Retrying connection to 10.10.1.2:5100 (attempt 5/60)...
  Retrying connection to 10.10.1.2:5100 (attempt 6/60)...
  Retrying connection to 10.10.1.2:5100 (attempt 7/60)...
  Retrying connection to 10.1

In [24]:
import pandas as pd

df_eff = pd.read_csv('/home/fabric/work/qkd-pqc-dependability/results/joint_fault_efficiency_loss.csv')
print(f"Total rows: {len(df_eff)}")
print(df_eff.groupby(['detector_efficiency', 'condition']).size().unstack(fill_value=0))

for c in ['qber', 'sifted_bits', 'key_bits_per_sec']:
    df_eff[c] = pd.to_numeric(df_eff[c], errors='coerce')

# Check for the classic contamination signature: duplicate elapsed_seconds across different conditions
dupes = df_eff[df_eff.duplicated(subset=['elapsed_seconds', 'sifted_bits'], keep=False)]
print(f"\nSuspicious duplicate rows: {len(dupes)}")
if len(dupes) > 0:
    print(dupes[['condition', 'detector_efficiency', 'run', 'elapsed_seconds', 'sifted_bits']])

Total rows: 84
condition            baseline  loss_15pct  loss_1pct  loss_5pct
detector_efficiency                                            
0.50                        9           9          9          9
0.70                        5           5          5          5
0.85                        4           4          4          4
1.00                        3           3          3          3

Suspicious duplicate rows: 0


In [26]:
import statsmodels.api as sm
from statsmodels.formula.api import ols

df_eff['loss_pct'] = df_eff['loss_pct'].fillna(0)
df_eff['loss_cat'] = df_eff['loss_pct'].astype(str)
df_eff['eff_cat'] = df_eff['detector_efficiency'].astype(str)

model_qber = ols('qber ~ C(eff_cat) + C(loss_cat) + C(eff_cat):C(loss_cat)', data=df_eff).fit()
print("=== QBER ANOVA (efficiency x loss) ===")
print(sm.stats.anova_lm(model_qber, typ=2))

model_rate = ols('key_bits_per_sec ~ C(eff_cat) + C(loss_cat) + C(eff_cat):C(loss_cat)', data=df_eff).fit()
print("\n=== Key Rate ANOVA (efficiency x loss) ===")
print(sm.stats.anova_lm(model_rate, typ=2))

=== QBER ANOVA (efficiency x loss) ===
                          sum_sq    df         F    PR(>F)
C(eff_cat)              0.000188   3.0  0.209884  0.889223
C(loss_cat)             0.000523   3.0  0.582865  0.628280
C(eff_cat):C(loss_cat)  0.001795   9.0  0.666439  0.736110
Residual                0.020349  68.0       NaN       NaN

=== Key Rate ANOVA (efficiency x loss) ===
                             sum_sq    df         F    PR(>F)
C(eff_cat)               117.009295   3.0  2.307904  0.084255
C(loss_cat)              133.220505   3.0  2.627656  0.057237
C(eff_cat):C(loss_cat)   134.268480   9.0  0.882775  0.545032
Residual                1149.185845  68.0       NaN       NaN
